# Librerias

In [ ]:
import uproot as up
import awkward as ak
import matplotlib.pyplot as plt
import vector as vec
import numpy as np
import pandas as pd
import os
from pathlib import Path
from scipy import stats as st
from matplotlib.ticker import MultipleLocator, FuncFormatter
from matplotlib.colors import LogNorm

vec.register_awkward()

In [ ]:
# Funciones

# Utilitary functions

def pi_formatter(x, pos):
    n = x / np.pi
    if np.isclose(n, 0):
        return r"$0$"
    if np.isclose(n, 1):
        return r"$\pi$"
    if np.isclose(n, -1):
        return r"$-\pi$"
    if n.is_integer():
        return rf"${int(n)}\pi$"
    return rf"${int(2*n)}\pi/2$"

def print_sample_info(Delphes, Energy_label, Gen_Events, Process, data_to_analyse, N_data):

    print("┌─────────────────────────────────────┐")
    print("│           SAMPLE INFORMATION        │")
    print("├─────────────────────────────────────┤")
    print(f"│ Delphes       : {Delphes:<20}│")
    print(f"│ Energy        : {Energy_label:<20}│")
    print(f"│ Gen. Events   : {Gen_Events:<20}│")
    print(f"│ Proc. Events  : {N_data:<20}│")
    print(f"│ Process       : {Process:<20}│")
    print("├─────────────────────────────────────┤")
    print(f"│ Dataset       : {data_to_analyse:<20}│")
    print("└─────────────────────────────────────┘")

def parse_events(s):
    s = s.strip().lower()
    mult = {"k": 1_000, "M": 1_000_000}
    if s[-1] in mult:
        return int(float(s[:-1]) * mult[s[-1]])
    return int(s)

def calculo_rangos(lim_inf, lim_sup, data):

    if type(data) == list:

        Gen_Events = sum(len(data_i) for data_i in data)

        debajo = sum(np.sum(data_i < lim_inf) for data_i in data)

        dentro = sum(
            np.sum((data_i >= lim_inf) & (data_i <= lim_sup))
            for data_i in data
        )

        encima = sum(np.sum(data_i > lim_sup) for data_i in data)

        p_debajo = 100 * debajo / Gen_Events
        p_dentro = 100 * dentro / Gen_Events
        p_encima = 100 * encima / Gen_Events

        return lim_inf, lim_sup, p_debajo, p_dentro, p_encima

    else:

        return print("Error: data debe ser una lista de arrays de datos.")

# Selection Functions

def MET_trigger(data, MET_lowerbond):

    MET_mask = ak.flatten(data["MissingET.MET"]) > MET_lowerbond

    return MET_mask

def preselection_cut(vec, Pt_lowerBond, Eta_absoluteBond):
    cinematic_mask = (vec.pt > Pt_lowerBond) & (abs(vec.eta) < Eta_absoluteBond)
    return cinematic_mask


def vectors_buildup(data):

    jets = ak.zip({
        "pt": data["Jet.PT"],
        "eta": data["Jet.Eta"],
        "phi": data["Jet.Phi"],
        "mass": data["Jet.Mass"],
        'TauTag': data["Jet.TauTag"],
        'BTag': data["Jet.BTag"],
    }, with_name="Momentum4D")

    tracks = ak.zip({"pt": data["Track.PT"], "eta": data["Track.Eta"],
                    "phi": data["Track.Phi"],
                    "mass": ak.zeros_like(data["Track.PT"]),
                    "charge": data["Track.Charge"],
                    "d0":  data["Track.D0"],
                    "dz":  data["Track.DZ"]},
                    with_name="Momentum4D")

    return jets, tracks


def Selected_data(data, MET_LB, PT_LB_jets, Eta_AB_jets, PT_LB_tracks, Eta_AB_tracks):
    MET_mask = MET_trigger(data, MET_LB)
    data_selected = data[MET_mask]

    jets, tracks = vectors_buildup(data_selected)

    jets_select   = jets[preselection_cut(jets,   PT_LB_jets,   Eta_AB_jets)]
    tracks_select = tracks[preselection_cut(tracks, PT_LB_tracks, Eta_AB_tracks)]

    return jets_select, tracks_select, MET_mask   

def Taus_and_bjets(jets):
    taus = jets[(jets['TauTag'] == 1) & (jets['BTag'] == 0)]
    bjets = jets[(jets['TauTag'] == 0) & (jets['BTag'] == 1)]

    N_events = len(ak.sum(jets, axis=1))
    N_taus = ak.num(taus, axis=1)
    N_bjets = ak.num(bjets, axis=1)

    return taus, bjets, N_events, N_taus, N_bjets

def kinematic_hierarchical_order(taus, bjets):
    mask  = (ak.num(taus) >= 2) & (ak.num(bjets) >= 2)
    taus_cut, bjets_cut = taus[mask], bjets[mask]
    tau1, tau2 = taus_cut[:,0], taus_cut[:,1]
    b1, b2     = bjets_cut[:,0], bjets_cut[:,1]
    return tau1, tau2, b1, b2, taus_cut, bjets_cut

def PT_and_IM_of_products(tau1, tau2, b1, b2):
    
    h_tautau_vec = tau1+tau2
    h_bb_vec = b1+b2
    
    M_tautau = h_tautau_vec.mass
    M_bb     = h_bb_vec.mass
    M_hh     = (tau1+tau2+b1+b2).mass

    PT_tautau = h_tautau_vec.pt
    PT_bb     = h_bb_vec.pt
    PT_hh     = (tau1+tau2+b1+b2).pt

    tuple_tautau = (M_tautau, PT_tautau)
    tuple_bb = (M_bb, PT_bb)
    tuple_hh = (M_hh, PT_hh)

    return tuple_tautau, tuple_bb, tuple_hh

# Data

In [ ]:
common_path_of_data = "/home/usuario/Madgraph/projects/"
saving_path = '/home/usuario/VSCode/diHiggs-Project/RecAnalysis/'

Delphes = "HL-LHC"
Energy_label = "14TeV"
BenchMark = "Pruebas"
process = "gg-hhj"

rutas = {
    'signal': "/home/usuario/Madgraph/projects/gg_hh-LQs/plusjet/output_ggHH_bbtata_BP1_80_0_delphes_events.root",
#    'bkg': "/home/usuario/Madgraph/projects/pp_tthj-1800k_reduced/output_pp_tthj_h-tata_1800k_104_0_delphes_events.root",
}

In [ ]:
truth_keys = ["Particle.PID", "Particle.Status",
              "Particle.Px", "Particle.Py", "Particle.Pz", "Particle.E",
              "Particle.X", "Particle.Y", "Particle.Z"]